# DATASET

Los datos los he recuperado desde internet. He descargado 383 imágenes de objetos que podemos llegar a ver en el día a día donde podemos ver la forma de una cara y 650 imágenes sin cara.

Creamos una funciones para renombrar las imágenes.

In [ ]:
import os

def renombrar_imagenes_cara(carpeta):
    extensiones_validas = {'.jpg', '.jpeg', '.png', '.gif', '.bmp', '.webp', '.tiff', '.svg'}

    archivos = [
        f for f in os.listdir(carpeta)
        if os.path.isfile(os.path.join(carpeta, f))
        and os.path.splitext(f)[1].lower() in extensiones_validas
    ]

    archivos.sort()

    # Comprobación previa: detectar si ya existen archivos con nomenclatura 'caraXXX'
    ya_renombrados = [f for f in archivos if f.lower().startswith('cara') and f[4:7].isdigit()]

    if ya_renombrados:
        print(f'⚠️  Se encontraron {len(ya_renombrados)} archivo(s) con nomenclatura "caraXXX":')
        for f in ya_renombrados:
            print(f'   - {f}')
        respuesta = input('/n¿Deseas continuar de todas formas? (s/n): ').strip().lower()
        if respuesta != 's':
            print('Operación cancelada.')
            return

    for i, archivo in enumerate(archivos, start=1):
        extension = os.path.splitext(archivo)[1].lower()
        nuevo_nombre = f'cara{i:03d}{extension}'
        origen = os.path.join(carpeta, archivo)
        destino = os.path.join(carpeta, nuevo_nombre)
        os.rename(origen, destino)
        print(f'{archivo} → {nuevo_nombre}')

    # print(f'\nListo. {len(archivos)} imágenes renombradas.')
    
def renombrar_imagenes_sincara(carpeta):
    extensiones_validas = {'.jpg', '.jpeg', '.png', '.gif', '.bmp', '.webp', '.tiff', '.svg'}

    archivos = [
        f for f in os.listdir(carpeta)
        if os.path.isfile(os.path.join(carpeta, f))
        and os.path.splitext(f)[1].lower() in extensiones_validas
    ]

    archivos.sort()

    # Comprobación previa: detectar si ya existen archivos con nomenclatura 'caraXXX'
    ya_renombrados = [f for f in archivos if f.lower().startswith('sin_cara') and f[4:7].isdigit()]

    if ya_renombrados:
        print(f'⚠️  Se encontraron {len(ya_renombrados)} archivo(s) con nomenclatura "sin_caraXXX":')
        for f in ya_renombrados:
            print(f'   - {f}')
        respuesta = input('/n¿Deseas continuar de todas formas? (s/n): ').strip().lower()
        if respuesta != 's':
            print('Operación cancelada.')
            return

    for i, archivo in enumerate(archivos, start=1):
        extension = os.path.splitext(archivo)[1].lower()
        nuevo_nombre = f'sin_cara{i:03d}{extension}'
        origen = os.path.join(carpeta, archivo)
        destino = os.path.join(carpeta, nuevo_nombre)
        os.rename(origen, destino)
        print(f'{archivo} → {nuevo_nombre}')

In [7]:
# Ejecutamos función desde los directorios donde tenemos las imágenes
cara_renombrar = 'C:/Users/urkow/Documents/Documentos_Clase/Caraduras_ML/img/train/cara'
sincara_renombrar = 'C:/Users/urkow/Documents/Documentos_Clase/Caraduras_ML/img/train/sin-cara'
# renombrar_imagenes_cara(cara_renombrar)
renombrar_imagenes_sincara(sincara_renombrar)

sin_cara001.jpeg → sin_cara001.jpeg
sin_cara002.jpg → sin_cara002.jpg
sin_cara003.jpg → sin_cara003.jpg
sin_cara004.jpg → sin_cara004.jpg
sin_cara005.jpg → sin_cara005.jpg
sin_cara006.jpg → sin_cara006.jpg
sin_cara007.jpg → sin_cara007.jpg
sin_cara008.jpg → sin_cara008.jpg
sin_cara009.jpg → sin_cara009.jpg
sin_cara010.png → sin_cara010.png
sin_cara011.png → sin_cara011.png
sin_cara012.jpg → sin_cara012.jpg
sin_cara013.jpg → sin_cara013.jpg
sin_cara014.jpg → sin_cara014.jpg
sin_cara015.jpg → sin_cara015.jpg
sin_cara016.jpg → sin_cara016.jpg
sin_cara017.jpg → sin_cara017.jpg
sin_cara018.jpg → sin_cara018.jpg
sin_cara019.jpg → sin_cara019.jpg
sin_cara020.png → sin_cara020.png
sin_cara021.jpg → sin_cara021.jpg
sin_cara022.jpg → sin_cara022.jpg
sin_cara023.jpeg → sin_cara023.jpeg
sin_cara024.jpg → sin_cara024.jpg
sin_cara025.jpg → sin_cara025.jpg
sin_cara026.png → sin_cara026.png
sin_cara027.jpeg → sin_cara027.jpeg
sin_cara028.jpg → sin_cara028.jpg
sin_cara029.jpg → sin_cara029.jpg
sin_cara

# Image EDA

In [12]:
# Imports básicos
import os
import random
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from glob import glob
import itertools
import json

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# Opcional: sklearn para métricas
try:
    from sklearn.metrics import confusion_matrix, classification_report, roc_auc_score
    from sklearn.utils.class_weight import compute_class_weight
    SKLEARN_AVAILABLE = True
except Exception:
    SKLEARN_AVAILABLE = False

# Parámetros globales
IMG_SIZE = (240, 240)
BATCH_SIZE = 32
AUTOTUNE = tf.data.AUTOTUNE
SEED = 42
tf.random.set_seed(SEED)
np.random.seed(SEED)

# Rutas de las imágenes
train_dir = '../img/train'
val_dir = '../img/val'
test_dir = '../img/test'

Vamos a ojear ciertas características de las imágenes, como tamaños, color, muestras, simetría, outliers

In [13]:
# Funciones utilitarias para EDA
def list_images(folder):
    return sorted(glob(os.path.join(folder, '*', '*')))

def sample_images(folder, n=20):
    imgs = list_images(folder)
    return random.sample(imgs, min(n, len(imgs)))

def load_pil(path):
    return Image.open(path).convert('RGB')

#### Distribución de tamaños y resoluciones

In [14]:
def analyze_sizes(paths):
    sizes = []
    for p in paths:
        try:
            with Image.open(p) as im:
                sizes.append(im.size)  # (width, height)
        except:
            continue
    sizes = np.array(sizes)
    return sizes

train_paths = list_images(train_dir)
val_paths = list_images(val_dir)
test_paths = list_images(test_dir)

sizes_train = analyze_sizes(train_paths)
print('Train images:', len(train_paths))
print('Unique sizes (sample):', np.unique(sizes_train, axis=0)[:10])

Train images: 1033
Unique sizes (sample): [[ 19 400]
 [ 24  24]
 [ 25  21]
 [ 46  46]
 [ 46  57]
 [ 48  46]
 [ 50  50]
 [ 69  46]
 [ 82  46]
 [101 400]]


#### Histograma de anchos y altos

In [ ]:
plt.figure(figsize=(10,4))
plt.subplot(1,2,1)
plt.hist(sizes_train[:,0], bins=30)
plt.title('Distribución de anchos (train)')
plt.subplot(1,2,2)
plt.hist(sizes_train[:,1], bins=30)
plt.title('Distribución de altos (train)')
plt.show()

# Color medio por imagen (RGB)
def mean_color(path):
    im = load_pil(path)
    arr = np.array(im).astype(np.float32)/255.0
    return arr.mean(axis=(0,1))

# calcular medias para una muestra (por rendimiento)
sample = random.sample(train_paths, min(300, len(train_paths)))
means = np.array([mean_color(p) for p in sample])
plt.figure(figsize=(6,4))
plt.scatter(means[:,0], means[:,1], c=means, s=20)
plt.xlabel('R mean'); plt.ylabel('G mean'); plt.title('Color medio (sample train)')
plt.show()

# Visualización aleatoria por clase
def show_random_by_class(folder, cls, n=9):
    paths = glob(os.path.join(folder, cls, '*'))
    paths = random.sample(paths, min(n, len(paths)))
    plt.figure(figsize=(8,8))
    for i,p in enumerate(paths):
        plt.subplot(3,3,i+1)
        plt.imshow(load_pil(p).resize((224,224)))
        plt.axis('off')
    plt.suptitle(f'{folder} / {cls} (muestra)')
    plt.show()

show_random_by_class('data/train', 'face', n=9)
show_random_by_class('data/train', 'no_face', n=9)

# Simetría horizontal (medida simple)
def symmetry_score(path):
    im = load_pil(path).resize(IMG_SIZE)
    arr = np.array(im).astype(np.float32)/255.0
    left = arr[:, :arr.shape[1]//2, :].mean(axis=2)
    right = arr[:, arr.shape[1] - arr.shape[1]//2:, :].mean(axis=2)[:, ::-1]
    # recortar a la misma forma si hay diferencia
    min_cols = min(left.shape[1], right.shape[1])
    left = left[:, :min_cols]
    right = right[:, :min_cols]
    return np.corrcoef(left.flatten(), right.flatten())[0,1]

# calcular simetría para una muestra por clase
sample_face = random.sample(glob('data/train/face/*'), min(200, len(glob('data/train/face/*'))))
sample_noface = random.sample(glob('data/train/no_face/*'), min(200, len(glob('data/train/no_face/*'))))

sym_face = np.array([symmetry_score(p) for p in sample_face])
sym_noface = np.array([symmetry_score(p) for p in sample_noface])

print('Simetría media face:', sym_face.mean(), 'no_face:', sym_noface.mean())
plt.boxplot([sym_face, sym_noface], labels=['face','no_face'])
plt.title('Distribución de simetría horizontal (muestra)')
plt.show()

# Detección rápida de outliers por tamaño o fallo de lectura
def find_outliers(paths):
    bad = []
    for p in paths:
        try:
            im = Image.open(p)
            w,h = im.size
            if w < 50 or h < 50:
                bad.append((p, 'too_small', (w,h)))
        except Exception as e:
            bad.append((p, 'error', str(e)))
    return bad

outliers = find_outliers(train_paths)
print('Outliers detectados (ejemplos):', outliers[:10])
